# TFM Deteccion - Ejecucion final

Pipeline definitivo: solo CNNDetection, 3 modelos (ResNet-50, ViT-B/16, UFD; FreqNet en stand-by), trainset de ejecucion 100.000 imgs (5000 por categoria, balance 50/50 real/fake) sobre las 20 categorias de progan_train. Validacion (progan_val) y test (progan_test + CNN_synth_testset, E1+E1b) completos.

El trainset se construye via API de Drive con streaming + range requests (NO via el mount FUSE) para evitar que Colab descargue el zip entero como cache local. Ver `problema_carga_datasets.md` para detalles.

Requisitos:
- Runtime Colab con GPU (T4 o superior).
- En `MyDrive/cnndetection-datasets/` (o como acceso directo): `progan_train.zip`, `progan_val.zip`, `progan_testset.zip`, `CNN_synth_testset.zip`.
- (Opcional) cuenta de wandb.


## 1. Drive + repo + dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!rm -rf /content/tfm_deteccion
!git clone https://github.com/manuelpalasanchez/tfm_deteccion.git
%cd /content/tfm_deteccion
!git pull

In [ ]:
!pip install -q -r requirements.txt

## 2. Inventario del progan_train.zip (opcional)

Lee el indice del zip directamente desde Drive (sin descomprimir nada). Ya hecho una vez: 20 cats x 36k imgs (50/50). Ejecutar solo si quieres reverificar.

In [ ]:
ZIP_TRAIN = '/content/drive/MyDrive/cnndetection-datasets/progan_train.zip'
!python scripts/scan_cnndetection.py --zip {ZIP_TRAIN} --csv reports/scan_progan_train.csv

## 3. Construccion del trainset de ejecucion (via Drive API)

Para evitar el problema de la cache FUSE de Drive (que descarga ~3.5 GB del zip por categoria accedida y llena el disco de Colab), usamos la API REST de Drive con range requests + streaming. Solo se transfiere el contenido de los ficheros seleccionados (~7-10 GB para N=5000), no el zip entero.

Las 20 categorias estan balanceadas (~36k imgs cada una), asi que muestreamos N fijo por categoria con balance 50/50 real-fake. Misma particion (seed=42) para los 3 modelos.


In [ ]:
# Autenticar con Drive API (no es lo mismo que drive.mount; este flujo no usa FUSE)
from google.colab import auth as colab_auth
colab_auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build

creds, _ = google.auth.default()
service = build('drive', 'v3', credentials=creds)

# Buscar el progan_train.zip (puede ser un acceso directo a otro Drive)
res = service.files().list(
    q="name='progan_train.zip' and trashed=false",
    fields="files(id,name,mimeType,size,shortcutDetails,owners)",
    supportsAllDrives=True,
    includeItemsFromAllDrives=True,
).execute()

matches = res.get('files', [])
if not matches:
    raise SystemExit('No se encontro ningun progan_train.zip accesible.')

print('Coincidencias encontradas:')
for f in matches:
    is_shortcut = f.get('mimeType') == 'application/vnd.google-apps.shortcut'
    target = f.get('shortcutDetails', {}).get('targetId') if is_shortcut else None
    size = f.get('size', '?')
    print('  id=' + f['id'] + '  shortcut=' + str(is_shortcut) + '  target=' + str(target) + '  size=' + str(size) + '  name=' + f['name'])

FILE_ID = matches[0]['id']
print('')
print('Usando FILE_ID = ' + FILE_ID)

N_PER_CAT = 5000
SEED = 42
OUT_TRAIN = '/content/cnndetection/progan_train'

!python scripts/build_trainset_drive_api.py --file-id {FILE_ID} --out {OUT_TRAIN} --n-per-cat {N_PER_CAT} --seed {SEED} --manifest reports/trainset_ejecucion_manifest.json


## 4. Extraccion de val + tests (completos)

progan_val (validacion durante el entrenamiento), progan_testset (E1) y CNN_synth_testset (E1b).

In [ ]:
import os, subprocess

DRIVE_ZIPS = '/content/drive/MyDrive/cnndetection-datasets'
CNN_ROOT = '/content/cnndetection'

for zip_name, dst in [
    ('progan_val.zip',        f'{CNN_ROOT}/progan_val'),
    ('progan_testset.zip',    CNN_ROOT),
    ('CNN_synth_testset.zip', CNN_ROOT),
]:
    print(f'extrayendo {zip_name}...')
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['unzip', '-q', '-n', f'{DRIVE_ZIPS}/{zip_name}', '-d', dst], check=True)

if os.path.isdir(f'{CNN_ROOT}/progan_testset') and not os.path.isdir(f'{CNN_ROOT}/progan_test'):
    os.rename(f'{CNN_ROOT}/progan_testset', f'{CNN_ROOT}/progan_test')
    print('renombrado progan_testset -> progan_test')

print('\nEstructura:')
!ls {CNN_ROOT}
!df -h /content | grep -v Filesystem

## 5. Patch de base.yaml

Apunta los roots a `/content/cnndetection`, habilita E1+E1b, deja E2 desactivado y guarda checkpoints en Drive para sobrevivir caidas de sesion.

In [ ]:
import yaml, pathlib

cfg_path = pathlib.Path('configs/base.yaml')
cfg = yaml.safe_load(cfg_path.read_text())

cfg['data']['train']['root']       = '/content/cnndetection'
cfg['data']['val']['root']         = '/content/cnndetection'
cfg['data']['eval_e1']['root']     = '/content/cnndetection'
cfg['data']['eval_e1']['split']    = 'test'
cfg['data']['eval_e1b']['root']    = '/content/cnndetection'
cfg['data']['eval_e1b']['enabled'] = True
cfg['data']['eval_e2']['enabled']  = False
cfg['data']['num_workers']         = 2

cfg['training']['epochs']     = 8
cfg['training']['batch_size'] = 128
cfg['training']['scheduler']['T_max'] = 8

cfg['output']['base_dir'] = '/content/drive/MyDrive/tfm-checkpoints'
cfg['wandb']['enabled'] = True

cfg_path.write_text(yaml.dump(cfg))
print(yaml.dump(cfg))

## 6. wandb (opcional)

Si no quieres tracking, salta esta celda y pon `cfg['wandb']['enabled'] = False` arriba.

In [ ]:
!wandb login

## 7bis. Fix de nombres post-extraccion (obligatorio)

El zip de progan_testset extrajo a una carpeta `progan/` y CNN_synth_testset extrajo todos los archs sueltos en `/content/cnndetection/`. Renombrar `progan -> progan_test` y mover los 12 archs a un root separado para que E1 (in-dist) y E1b (cross-arch) carguen datasets distintos.


In [ ]:
!mkdir -p /content/CNN_synth_testset
!mv /content/cnndetection/progan /content/cnndetection/progan_test 2>/dev/null || echo 'progan ya renombrado'
!for d in biggan crn cyclegan deepfake gaugan imle san seeingdark stargan stylegan stylegan2 whichfaceisreal; do mv /content/cnndetection/$d /content/CNN_synth_testset/${d}_test 2>/dev/null; done

import yaml, pathlib
p = pathlib.Path('configs/base.yaml')
c = yaml.safe_load(p.read_text())
c['data']['eval_e1b']['root'] = '/content/CNN_synth_testset'
p.write_text(yaml.dump(c))

print('--- /content/cnndetection ---')
!ls /content/cnndetection
print('--- /content/CNN_synth_testset ---')
!ls /content/CNN_synth_testset


## 7. Sanity check

In [ ]:
import sys
sys.path.insert(0, '/content/tfm_deteccion')
from data.cnndetection_dataset import CNNDetectionDataset
from data.transforms import get_eval_transforms

for split in ('train', 'val', 'test'):
    try:
        ds = CNNDetectionDataset(root='/content/cnndetection', split=split, transform=get_eval_transforms())
        print(f'{split}: {len(ds)} muestras, generadores={ds.generators}')
    except Exception as e:
        print(f'{split}: ERROR - {e}')

# Cross-arch (E1b) lo carga el evaluator desde /content/CNN_synth_testset
try:
    ds_e1b = CNNDetectionDataset(root='/content/CNN_synth_testset', split='test', transform=get_eval_transforms())
    print(f'e1b: {len(ds_e1b)} muestras, generadores={ds_e1b.generators}')
except Exception as e:
    print(f'e1b: ERROR - {e}')


## 8. Train + evaluate (modelo a modelo)

Tres bloques: ResNet-50, ViT-B/16, UniversalFakeDetect. Cada bloque tiene una celda de train y otra de eval. Los checkpoints se guardan en `/content/drive/MyDrive/tfm-checkpoints/{modelo}_<timestamp>/` y la celda de eval localiza automaticamente el ultimo run.

**Recuperacion**: si un modelo ya esta entrenado en una sesion anterior (su carpeta tiene `checkpoint_best.pth`), saltar la celda de train y correr solo la de eval. Si la carpeta tambien tiene `metrics.json` y los PNGs, ese modelo esta completo y se puede saltar entero.

Configs ya parcheados:
- `configs/vit.yaml`: `batch_size=64` (ViT-B/16 no entra a 128 en T4).
- `configs/universalfakedetect.yaml`: `epochs=3` (CLIP frozen, la cabeza converge rapido y 8 epocas no caben en una sesion).


### ResNet-50


In [ ]:
# Train ResNet-50 (~50 min en T4)
!python scripts/train.py --config configs/resnet50.yaml


In [ ]:
# Eval ResNet-50: localiza el ultimo run y corre evaluate.py (~5 min)
import glob
modelo = 'resnet50'
ckpts = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints/{modelo}_*/checkpoint_best.pth'))
assert ckpts, f'Sin checkpoint para {modelo}'
ckpt = ckpts[-1]
print('checkpoint:', ckpt)
!python scripts/evaluate.py --config configs/{modelo}.yaml --checkpoint "{ckpt}"


### ViT-B/16


In [ ]:
# Train ViT-B/16 (~2-2.5 h en T4 con batch=64)
!python scripts/train.py --config configs/vit.yaml


In [ ]:
# Eval ViT-B/16
import glob
modelo = 'vit'
ckpts = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints/{modelo}_*/checkpoint_best.pth'))
assert ckpts, f'Sin checkpoint para {modelo}'
ckpt = ckpts[-1]
print('checkpoint:', ckpt)
!python scripts/evaluate.py --config configs/{modelo}.yaml --checkpoint "{ckpt}"


### UniversalFakeDetect


In [ ]:
# Train UFD (~3-4 h en T4; con CLIP frozen + 3 epocas configuradas)
!python scripts/train.py --config configs/universalfakedetect.yaml


In [ ]:
# Eval UFD
import glob
modelo = 'universalfakedetect'
ckpts = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints/{modelo}_*/checkpoint_best.pth'))
assert ckpts, f'Sin checkpoint para {modelo}'
ckpt = ckpts[-1]
print('checkpoint:', ckpt)
!python scripts/evaluate.py --config configs/{modelo}.yaml --checkpoint "{ckpt}"


## 9. Resumen de metricas

In [ ]:
import glob, json, csv, os

CKPT_BASE = '/content/drive/MyDrive/tfm-checkpoints'
modelos = ['resnet50', 'vit', 'universalfakedetect']
rows = []

for modelo in modelos:
    dirs = sorted(glob.glob(f'{CKPT_BASE}/{modelo}_*/metrics.json'))
    if not dirs:
        print(f'{modelo}: sin metrics.json todavia')
        continue
    metrics_path = dirs[-1]
    with open(metrics_path) as f:
        m = json.load(f)
    row = {'modelo': modelo}
    for ronda, vals in m.items():
        row[f'{ronda}_auc']  = round(vals.get('auc_roc', float('nan')), 4)
        row[f'{ronda}_ap']   = round(vals.get('average_precision', float('nan')), 4)
        row[f'{ronda}_acc']  = round(vals.get('accuracy', float('nan')), 4)
    rows.append(row)
    print(f'{modelo}: {metrics_path}')
    for k, v in row.items():
        if k != 'modelo':
            print(f'  {k}: {v}')

if rows:
    csv_path = f'{CKPT_BASE}/resumen_metricas.csv'
    fieldnames = list(rows[0].keys())
    with open(csv_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f'\nCSV guardado en {csv_path}')


---
## 10. Figuras comparativas

Genera todas las figuras de la Sección 8 de la memoria directamente en `MyDrive/tfm-checkpoints/figuras_memoria/`. No requiere re-entrenamiento — las métricas globales se leen de los `metrics.json` ya guardados en Drive.

La parte de inferencia E1b (~5 min por modelo en T4) genera el desglose por arquitectura GAN, los archivos `.pt` de predicciones, las curvas ROC superpuestas y el heatmap.


In [ ]:
import json, csv, importlib
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import seaborn as sns

plt.rcParams.update({'font.family': 'serif', 'axes.spines.top': False, 'axes.spines.right': False})

CKPT_BASE   = '/content/drive/MyDrive/tfm-checkpoints'
FIGURAS_DIR = Path(CKPT_BASE) / 'figuras_memoria'
FIGURAS_DIR.mkdir(parents=True, exist_ok=True)

modelo_folders = {
    'ResNet-50':           'resnet50_20260508_115724',
    'ViT-B/16':            'vit_20260509_122527',
    'UniversalFakeDetect': 'universalfakedetect_20260509_153838',
}
MODELOS = list(modelo_folders.keys())
COLORES = {'ResNet-50': '#1976D2', 'ViT-B/16': '#388E3C', 'UniversalFakeDetect': '#F57C00'}

resultados   = {}
matrices_e1b = {}

for nombre, folder in modelo_folders.items():
    mpath = Path(CKPT_BASE) / folder / 'metrics.json'
    with mpath.open() as f:
        m = json.load(f)
    e1b = m['eval_e1b']
    resultados[nombre] = {
        'e1':  {'auc': m['eval_e1']['auc_roc'], 'ap': m['eval_e1']['average_precision'], 'acc': m['eval_e1']['accuracy']},
        'e1b': {'auc': e1b['auc_roc'], 'ap': e1b['average_precision'], 'acc': e1b['accuracy'], 'n': e1b.get('n_samples', 0)},
    }
    cm_val = e1b.get('confusion_matrix')
    if cm_val:
        matrices_e1b[nombre] = cm_val

print(f'FIGURAS_DIR: {FIGURAS_DIR}')
for nombre, vals in resultados.items():
    auc = vals['e1b']['auc']
    ap  = vals['e1b']['ap']
    acc = vals['e1b']['acc']
    n   = vals['e1b']['n']
    print(f'  {nombre}: E1b AUC={auc:.4f}  AP={ap:.4f}  Acc={acc:.4f}  n={n}')


In [ ]:
# Figura 1 — Matrices de confusion E1b comparativa
if not matrices_e1b:
    print('AVISO: confusion_matrix no en metrics.json. Re-evalua con metrics.py actualizado.')
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    tick_labels = ['Real', 'Fake']

    for ax, modelo in zip(axes, MODELOS):
        cm = np.array(matrices_e1b[modelo])
        cm_norm = cm.astype(float) / cm.max()
        sns.heatmap(cm, annot=False, cmap='Blues',
                    xticklabels=tick_labels, yticklabels=tick_labels,
                    ax=ax, cbar=False, linewidths=0.5, linecolor='white')
        for i in range(2):
            for j in range(2):
                val = cm[i, j]
                val_str = f'{val:,}'.replace(',', '.')
                text_color = 'white' if cm_norm[i, j] > 0.5 else 'black'
                ax.text(j + 0.5, i + 0.5, val_str,
                        ha='center', va='center', fontsize=13, fontweight='bold', color=text_color)
        ax.set_title(modelo, fontsize=13, pad=10)
        ax.set_xlabel('Predicho', fontsize=11)
        ax.set_ylabel('Clase real', fontsize=11)
        ax.tick_params(axis='both', labelsize=10)

    n_total = resultados[MODELOS[0]]['e1b']['n']
    n_str = f'{n_total:,}'.replace(',', '.')
    plt.suptitle(
        f'Matrices de confusion — Evaluacion E1b (cross-architecture, {n_str} imagenes)',
        fontsize=14, y=1.03,
    )
    plt.tight_layout()

    out_path = FIGURAS_DIR / 'matrices_confusion_e1b_comparativa.png'
    plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Guardado: {out_path}')


In [ ]:
# Figura 2 — Barras de metricas E1b
fig, ax = plt.subplots(figsize=(10, 6))

metricas_keys   = ['auc',     'ap',     'acc']
metricas_labels = ['AUC-ROC', 'AP',     'Accuracy']
bar_colors      = ['#1976D2', '#388E3C', '#F57C00']
width   = 0.25
x       = np.arange(len(MODELOS))
offsets = [-width, 0.0, width]

for key, label, color, offset in zip(metricas_keys, metricas_labels, bar_colors, offsets):
    valores = [resultados[m]['e1b'][key] for m in MODELOS]
    bars = ax.bar(x + offset, valores, width, label=label, color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, valores):
        fmt_str = f'{val * 100:.2f}%' if key == 'acc' else f'{val:.4f}'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                fmt_str, ha='center', va='bottom', fontsize=8.5)

ax.set_ylim(0.70, 1.03)
ax.set_xticks(x)
ax.set_xticklabels(MODELOS, fontsize=11)
ax.set_ylabel('Valor de la metrica', fontsize=12)
ax.set_title('Comparativa de metricas — Evaluacion E1b (cross-architecture)', fontsize=13)
ax.legend(fontsize=11, loc='lower right')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()

out_path = FIGURAS_DIR / 'barras_metricas_e1b.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path}')


In [ ]:
# Figura 3B — Curvas ROC en paneles (PNGs del evaluador)
roc_pngs = {m: Path(CKPT_BASE) / folder / 'roc_curve_e1b.png'
            for m, folder in modelo_folders.items()}

for modelo, ruta in roc_pngs.items():
    print(f'  {modelo}: {"OK" if ruta.exists() else "NO ENCONTRADO"}')

fig, axes = plt.subplots(1, 3, figsize=(21, 7))
for ax, (modelo, png_path) in zip(axes, roc_pngs.items()):
    img = mpimg.imread(str(png_path))
    ax.imshow(img)
    ax.set_title(modelo, fontsize=14, pad=10)
    ax.axis('off')

plt.suptitle('Curvas ROC — Evaluacion E1b (cross-architecture)', fontsize=15, y=1.01)
plt.tight_layout()

out_path = FIGURAS_DIR / 'roc_curves_e1b_paneles.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path}')


In [ ]:
# Inferencia E1b — un pase por modelo (~5 min por modelo en T4)
import sys
sys.path.insert(0, '/content/tfm_deteccion')

import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_curve, auc as sklearn_auc
from tqdm import tqdm

import data.cnndetection_dataset
from models import model_registry
from utils.config import load_config
from data.cnndetection_dataset import CNNDetectionDataset
from data.transforms import get_eval_transforms
from evaluation.metrics import compute_metrics

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
E1B_ROOT = '/content/CNN_synth_testset'
print(f'Device: {DEVICE}')

modelos_3a = {
    'ResNet-50': {
        'config':     Path('/content/tfm_deteccion/configs/resnet50.yaml'),
        'checkpoint': Path(CKPT_BASE) / 'resnet50_20260508_115724' / 'checkpoint_best.pth',
        'module':     'models.resnet50',
        'csv_key':    'resnet50',
        'color':      '#1976D2',
    },
    'ViT-B/16': {
        'config':     Path('/content/tfm_deteccion/configs/vit.yaml'),
        'checkpoint': Path(CKPT_BASE) / 'vit_20260509_122527' / 'checkpoint_best.pth',
        'module':     'models.vit',
        'csv_key':    'vit',
        'color':      '#388E3C',
    },
    'UniversalFakeDetect': {
        'config':     Path('/content/tfm_deteccion/configs/universalfakedetect.yaml'),
        'checkpoint': Path(CKPT_BASE) / 'universalfakedetect_20260509_153838' / 'checkpoint_best.pth',
        'module':     'models.universalfakedetect',
        'csv_key':    'ufd',
        'color':      '#F57C00',
    },
}

print('Checkpoints:')
for nombre, info in modelos_3a.items():
    estado = 'OK' if info['checkpoint'].exists() else 'NO ENCONTRADO'
    print(f'  {nombre}: {estado}')
n_arqs = len(list(Path(E1B_ROOT).glob('*_test'))) if Path(E1B_ROOT).exists() else 0
print(f'E1B_ROOT: {E1B_ROOT}  ({n_arqs} arquitecturas GAN)')


In [ ]:
desglose_data = {}
roc_data      = {}

for nombre, info in modelos_3a.items():
    csv_key = info['csv_key']
    print('\n' + '='*55)
    print(f'  Modelo: {nombre}')
    print('='*55)

    cfg = load_config(info['config'])
    importlib.import_module(info['module'])
    model = model_registry.build(cfg.model.name, **vars(cfg.model.kwargs))
    ckpt = torch.load(str(info['checkpoint']), map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    best_auc = ckpt.get('best_auc', float('nan'))
    print(f'  Checkpoint cargado (val best_auc={best_auc:.4f})')

    dataset = CNNDetectionDataset(root=E1B_ROOT, split='test', transform=get_eval_transforms())
    loader  = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
    print(f'  Dataset E1b: {len(dataset)} imagenes')

    all_scores, all_targets, all_generators = [], [], []
    with torch.no_grad():
        for images, labels, generators in tqdm(loader, desc='  Inferencia'):
            images = model.preprocess(images.to(DEVICE))
            logits = model(images).squeeze(1)
            scores = torch.sigmoid(logits).cpu().numpy()
            all_scores.extend(scores.tolist())
            all_targets.extend(labels.numpy().tolist())
            all_generators.extend(generators)

    y_true  = np.array(all_targets)
    y_score = np.array(all_scores)
    gens    = np.array(all_generators)

    pt_path = FIGURAS_DIR / f'{csv_key}_e1b_predictions.pt'
    torch.save({'y_true':  torch.tensor(y_true,  dtype=torch.float32),
                'y_score': torch.tensor(y_score, dtype=torch.float32),
                'generators': list(gens)}, str(pt_path))
    print(f'  Predictions guardadas: {pt_path}')

    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc_global  = sklearn_auc(fpr, tpr)
    roc_data[nombre] = {'fpr': fpr, 'tpr': tpr, 'auc': auc_global, 'color': info['color']}
    print(f'  AUC-ROC global E1b: {auc_global:.4f}')

    arqs = sorted(set(gens))
    print(f'  Arquitecturas ({len(arqs)}): {", ".join(arqs)}')
    print('  ' + '-'*55)
    for arch in arqs:
        mask = gens == arch
        m = compute_metrics(y_true[mask], y_score[mask])
        if arch not in desglose_data:
            desglose_data[arch] = {'n_samples': int(mask.sum())}
        desglose_data[arch][f'{csv_key}_auc'] = m['auc_roc']
        desglose_data[arch][f'{csv_key}_ap']  = m['average_precision']
        desglose_data[arch][f'{csv_key}_acc'] = m['accuracy']
        auc_v = m['auc_roc']
        ap_v  = m['average_precision']
        acc_v = m['accuracy']
        print(f'  {arch:25s}  AUC={auc_v:.4f}  AP={ap_v:.4f}  Acc={acc_v:.4f}')

    del model
    torch.cuda.empty_cache()

print(f'\nInferencia completada: {list(roc_data.keys())}')


In [ ]:
CSV_COLS = [
    'arquitectura_gan', 'n_samples',
    'resnet50_auc', 'resnet50_ap', 'resnet50_acc',
    'vit_auc',      'vit_ap',      'vit_acc',
    'ufd_auc',      'ufd_ap',      'ufd_acc',
]

csv_path = FIGURAS_DIR / 'desglose_e1b_por_arquitectura.csv'
with csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLS, extrasaction='ignore')
    writer.writeheader()
    for arch in sorted(desglose_data.keys()):
        row = {'arquitectura_gan': arch, **desglose_data[arch]}
        writer.writerow(row)

print(f'CSV guardado: {csv_path}\n')
print('Arquitectura GAN           N      ResNet AUC   ViT AUC   UFD AUC')
print('-' * 65)
for arch in sorted(desglose_data.keys(), key=lambda a: -desglose_data[a].get('resnet50_auc', 0)):
    d = desglose_data[arch]
    n     = d['n_samples']
    r_auc = d.get('resnet50_auc', float('nan'))
    v_auc = d.get('vit_auc',      float('nan'))
    u_auc = d.get('ufd_auc',      float('nan'))
    print(f'{arch:25s}  {n:>6}  {r_auc:>10.4f}  {v_auc:>9.4f}  {u_auc:>9.4f}')


In [ ]:
# Figura 3A — Curvas ROC E1b superpuestas
fig, ax = plt.subplots(figsize=(8, 8))
for nombre, rd in roc_data.items():
    rd_auc = rd['auc']
    ax.plot(rd['fpr'], rd['tpr'], color=rd['color'], lw=2.5,
            label=f'{nombre} (AUC = {rd_auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Clasificador aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=13)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=13)
ax.set_title('Curvas ROC — Evaluacion E1b (cross-architecture)', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.01])
ax.set_aspect('equal')
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()

out_path = FIGURAS_DIR / 'roc_curves_e1b_superpuestas.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path}')


In [ ]:
# Heatmap AUC-ROC por arquitectura GAN x modelo detector + resumen de archivos
csvkeys_hm  = ['resnet50_auc', 'vit_auc', 'ufd_auc']
arqs_sorted = sorted(
    desglose_data.keys(),
    key=lambda a: -np.mean([desglose_data[a].get(k, 0.0) for k in csvkeys_hm]),
)

heat_matrix = np.array([
    [desglose_data[arch].get(k, np.nan) for k in csvkeys_hm]
    for arch in arqs_sorted
])

fig, ax = plt.subplots(figsize=(8, max(4, len(arqs_sorted) * 0.8)))
sns.heatmap(
    heat_matrix, annot=False, cmap='RdYlGn', vmin=0.5, vmax=1.0,
    xticklabels=MODELOS, yticklabels=arqs_sorted,
    ax=ax, linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'AUC-ROC', 'shrink': 0.6},
)

for i, arch in enumerate(arqs_sorted):
    for jj, key in enumerate(csvkeys_hm):
        val = desglose_data[arch].get(key, np.nan)
        if not np.isnan(val):
            norm = (val - 0.5) / 0.5
            text_color = 'white' if norm < 0.15 or norm > 0.85 else 'black'
            ax.text(jj + 0.5, i + 0.5, f'{val:.4f}',
                    ha='center', va='center', fontsize=9, fontweight='bold', color=text_color)

ax.set_title('AUC-ROC por arquitectura generativa y modelo detector', fontsize=13, pad=12)
ax.set_xlabel('Modelo detector', fontsize=12)
ax.set_ylabel('Arquitectura GAN (ordenada por AUC media)', fontsize=12)
ax.tick_params(axis='x', rotation=15, labelsize=10)
ax.tick_params(axis='y', rotation=0,  labelsize=10)
plt.tight_layout()

out_path = FIGURAS_DIR / 'heatmap_auc_por_generador.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path}')

print(f'\nFiguras en {FIGURAS_DIR}:')
for archivo in sorted(FIGURAS_DIR.iterdir()):
    size_kb = archivo.stat().st_size / 1024
    print(f'  {archivo.name:55s}  {size_kb:>7.1f} KB')
